# 01. Exploratory Data Analysis & Data Preprocessing
Sistem Segmentasi Jenis Usaha UMKM DKI Jakarta Berbasis Wilayah Menggunakan Algoritma K-Modes Clustering  

---

### **Tujuan Notebook:**
1. Memuat dataset mentah UMKM DKI Jakarta (`.xls` berformat tabel HTML).
2. Melakukan audit kualitas data (*null values*, tipe data, dan ukuran data).
3. Melakukan prapemrosesan dan pembersihan baris data yang cacat/kosong pada atribut krusial (`jenis_usaha`), difilter pada `periode_data == 2024` (snapshot kumulatif 2020-2024, ~266.678 baris bersih).
4. Merekayasa 3 fitur kategorikal untuk K-Modes clustering: `sektor_makro`, `jenis_instansi_pembina`, dan `kelompok_tahun_berdiri`.
5. Menyimpan dataset bersih ke `data/processed/01_umkm_clean.csv`.

> **Catatan periode data:** `periode_data == 2025` hanya berisi usaha yang baru terdaftar pada tahun 2025 (kolom `tahun` konstan = 2025 untuk seluruh barisnya), sehingga tidak bisa dipakai untuk fitur `kelompok_tahun_berdiri`. `periode_data == 2024` adalah snapshot kumulatif 2020-2024 dengan variasi `tahun` yang nyata, sehingga dipakai sebagai sumber data pada versi ini.

In [1]:
import os
import pandas as pd
import numpy as np

# Pengaturan opsi tampilan Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Pustaka sukses dimuat.")

Pustaka sukses dimuat.


## 1. Input Dataset Mentah
Karena file `.xls` tersebut terstruktur dalam format tabel HTML, pembacaan dilakukan menggunakan fungsi `pd.read_html()`.

In [2]:
# Path lokasi file mentah
raw_data_path = os.path.join('..', 'data', 'raw', 'Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls')

# Jika notebook dijalankan langsung di direktori utama (root), sesuaikan path:
if not os.path.exists(raw_data_path):
    raw_data_path = 'Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls'

print(f"Membaca berkas dari: {raw_data_path}")
tables = pd.read_html(raw_data_path)
df_raw = tables[0]

print(f"Dimensi Data Mentah (Baris, Kolom): {df_raw.shape}")
df_raw.head(5)

Membaca berkas dari: ..\data\raw\Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls
Dimensi Data Mentah (Baris, Kolom): (298893, 7)


,periode_data,wilayah,skpd,jenis_usaha,nama_usaha,tahun,nama_pengusaha
0,2024,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,SABLON DAN DESIGN GRAFIS,RADH DESIGN,2020,RADHIYYAH MARDHIYYAH
1,2024,KOTA ADM. JAKARTA SELATAN,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RADIMSUM,2021,LIDIA NASTALIA
2,2024,KAB. ADM. KEP. SERIBU,DINAS PARIWISATA DAN EKONOMI KREATIF,AGEN WISATA PERJALANAN (AWP),RADWA WISATA,2020,EVA OKTAVIANI
3,2024,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RAFARIFOOD_IKANSEGAR,2020,CARIDI
4,2024,KOTA ADM. JAKARTA TIMUR,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RAFIANTUN UDUK,2022,RAFIATUN


## 2. Audit Kualitas Data
Langkah ini bertujuan untuk mengidentifikasi keberadaan nilai hilang (*missing values*) serta kesesuaian tipe data pada setiap atribut.

In [3]:
audit_summary = pd.DataFrame({
    'Tipe Data': df_raw.dtypes,
    'Missing Value': df_raw.isnull().sum(),
    'Persentase Missing (%)': (df_raw.isnull().sum() / len(df_raw)) * 100,
    'Kategori Unik': df_raw.nunique()
})

print("=== RINGKASAN AUDIT DATA MENTAH ===")
audit_summary

=== RINGKASAN AUDIT DATA MENTAH ===


,Tipe Data,Missing Value,Persentase Missing (%),Kategori Unik
periode_data,int64,0,0.000000,2
wilayah,str,0,0.000000,6
skpd,str,0,0.000000,49
jenis_usaha,str,18,0.006022,161
nama_usaha,str,183,0.061226,228496
tahun,int64,0,0.000000,6
nama_pengusaha,str,0,0.000000,176619


## 3. Pembersihan Data (*Data Cleaning*)
Sesuai rancangan metodologi, atribut `jenis_usaha` merupakan variabel kategorikal utama. Baris data yang memiliki nilai `NaN` pada kolom `jenis_usaha` akan dieliminasi. Data difilter pada `periode_data == 2024` karena periode ini adalah snapshot kumulatif 2020-2024 dengan variasi `tahun` yang nyata (dibutuhkan agar fitur `kelompok_tahun_berdiri` bermakna), berbeda dengan `periode_data == 2025` yang hanya memuat usaha baru terdaftar tahun itu saja.

In [4]:
# 1. Bersihkan missing values pada jenis_usaha
df_clean = df_raw.dropna(subset=['jenis_usaha']).copy()

# 2. Filter periode tahun 2024 (snapshot kumulatif, bukan 2025 yang cuma usaha baru)
if 'periode_data' in df_clean.columns:
    df_clean = df_clean[df_clean['periode_data'].astype(str).str.contains('2024')].copy()
elif 'tahun' in df_clean.columns:
    df_clean = df_clean[df_clean['tahun'].astype(str).str.contains('2024')].copy()

# 3. Hapus data duplikat (Drop Duplicates)
baris_sebelum_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
jumlah_duplikat = baris_sebelum_dedup - len(df_clean)

# 4. Standardisasi teks string
for col in ['wilayah', 'skpd', 'jenis_usaha']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

print("=== HASIL PEMBERSIHAN, FILTER 2024 & DEDUP ===")
print(f"Total baris mentah awal         : {len(df_raw):,} baris")
print(f"Total baris periode 2024        : {baris_sebelum_dedup:,} baris")
print(f"Baris duplikat tereliminasi     : {jumlah_duplikat:,} baris")
print(f"Total baris bersih unik akhir   : {len(df_clean):,} baris")

=== HASIL PEMBERSIHAN, FILTER 2024 & DEDUP ===
Total baris mentah awal         : 298,893 baris
Total baris periode 2024        : 266,681 baris
Baris duplikat tereliminasi     : 3 baris
Total baris bersih unik akhir   : 266,678 baris


## 4. Fitur 1 — Pemetaan 155 `jenis_usaha` ke dalam 6 Sektor Makro
Pemetaan `jenis_usaha` ke `sektor_makro` dilakukan lewat *lookup dictionary* eksak (bukan pencocokan kata kunci/*keyword matching*), karena seluruh 155 nilai unik `jenis_usaha` pada periode 2024 (dan 2025) sudah diketahui dan terverifikasi termapping tanpa sisa. Pendekatan ini menghindari salah-klasifikasi yang bisa terjadi pada pencocokan kata kunci (mis. kata yang muncul di lebih dari satu kategori).

In [5]:
SEKTOR_MAP = {
 # Kuliner & Pangan Olahan
 'AIR MINUM ISI ULANG':'Kuliner & Pangan Olahan','CAFE / RUMAH MINUM':'Kuliner & Pangan Olahan',
 'DODOL':'Kuliner & Pangan Olahan','INDUSTRI BUMBU MASAK DAN PENYEDAP MASAKAN':'Kuliner & Pangan Olahan',
 'INDUSTRI PRODUK MASAK LAINNYA':'Kuliner & Pangan Olahan','JASA CATERING':'Kuliner & Pangan Olahan',
 'KECAP, TAHU ,TEMPE, PEYEK KACANG, KACANG SUKRO, MAYONAISE DAN SEJENISNYA':'Kuliner & Pangan Olahan',
 'KEDAI JAMU':'Kuliner & Pangan Olahan','KEDAI MAKANAN':'Kuliner & Pangan Olahan',
 'KERUPUK, KERIPIK, PEYEK DAN SEJENISNYA':'Kuliner & Pangan Olahan','KUE BASAH':'Kuliner & Pangan Olahan',
 'MAKANAN BAYI':'Kuliner & Pangan Olahan','MAKANAN DAN MINUMAN':'Kuliner & Pangan Olahan',
 'PENJUAL MAKANAN KELILING/ GEROBAKAN':'Kuliner & Pangan Olahan','PENJUAL MINUMAN KELILING/ GEROBAKAN':'Kuliner & Pangan Olahan',
 'PRODUK MAKANAN LAINNYA':'Kuliner & Pangan Olahan','RESTORAN':'Kuliner & Pangan Olahan',
 'RUMAH/WARUNG MAKAN':'Kuliner & Pangan Olahan','TAHU KEDELAI':'Kuliner & Pangan Olahan',
 'TEMPE KEDELAI':'Kuliner & Pangan Olahan','WARUNG KOPI / KEDAI KOPI , KEDAI JUS DAN MINUMAN LAINNYA':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI MAKANAN DAN MINUMAN YTDL':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR MINUMAN':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR ROTI, KUE KERING, KUE BASAH DAN SEJENISNYA':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR TAHU, TEMPE, TAUCO DAN ONCOM':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN MAKANAN LAINNYA':'Kuliner & Pangan Olahan','PERDAGANGAN ECERAN MINUMAN TIDAK BERALKOHOL':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN ROTI, KUE KERING, SERTA KUE BASAH DAN SEJENISNYA':'Kuliner & Pangan Olahan',
 'PERDAGANGAN ECERAN TAHU, TEMPE, TAUCO DAN ONCOM':'Kuliner & Pangan Olahan',

 # Fesyen & Industri Tekstil
 'FASHION':'Fesyen & Industri Tekstil','KONVEKSI PAKAIAN DARI KULIT':'Fesyen & Industri Tekstil',
 'KONVEKSI PAKAIAN DARI TEKSTIL':'Fesyen & Industri Tekstil','MENJAHIT / BORDIR':'Fesyen & Industri Tekstil',
 'VERMAK PAKAIAN':'Fesyen & Industri Tekstil','PEDAGANG KAKI LIMA DAN LOS PASAR - PELENGKAP PAKAIAN DAN BENANG':'Fesyen & Industri Tekstil',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO - PAKAIAN':'Fesyen & Industri Tekstil',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO - SEPATU, SANDAL DAN ALAS KAKI LAINNYA':'Fesyen & Industri Tekstil',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO- TEKSTIL':'Fesyen & Industri Tekstil',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-BARANG PERHIASAN':'Fesyen & Industri Tekstil',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-KOSMETIK':'Fesyen & Industri Tekstil',

 # Perdagangan Sembako & Retail
 'AGEN LPG ECERAN &PERTAMINI':'Perdagangan Sembako & Retail','ALAT TULIS KANTOR DAN GAMBAR':'Perdagangan Sembako & Retail',
 'BARANG PECAH BELAH DAN PERLENGKAPAN DAPUR':'Perdagangan Sembako & Retail','BUKU DAN HASIL PENCETAKAN':'Perdagangan Sembako & Retail',
 'ERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOPI, GULA PASIR, GULA MERAH DAN SEJENISNYA':'Perdagangan Sembako & Retail',
 'INDUSTRI SABUN DAN BAHAN PEMBERSIH KEPERLUAN RUMAH TANGGA':'Perdagangan Sembako & Retail',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-AROMATIK/ PENYEGAR':'Perdagangan Sembako & Retail',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-FARMASI':'Perdagangan Sembako & Retail',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-OBAT TRADISIONAL':'Perdagangan Sembako & Retail',
 'PEMBUNGKUS DARI PLASTIK':'Perdagangan Sembako & Retail','PERALATAN LISTRIK RUMAH TANGGA, PERALATAN PENERANGAN DAN PERLENGKAPANNYA':'Perdagangan Sembako & Retail',
 'PERALATAN OLAH RAGA':'Perdagangan Sembako & Retail','PERDAGANGAN ECERAN BERAS':'Perdagangan Sembako & Retail',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR BERAS':'Perdagangan Sembako & Retail',
 'PERDAGANGAN ECERAN KOPI, GULA PASIR DAN GULA MERAH':'Perdagangan Sembako & Retail',
 'PERMAINAN DAN ALAT MAINAN ANAK':'Perdagangan Sembako & Retail','PULSA':'Perdagangan Sembako & Retail',
 'TOKO BARANG KELONTONG':'Perdagangan Sembako & Retail','WARUNG / TOKO BAHAN KEBUTUHAN POKOK':'Perdagangan Sembako & Retail',
 'WARUNG SEMBAKO':'Perdagangan Sembako & Retail',

 # Agrobisnis, Perikanan & Hayati
 'AGROBISNIS':'Agrobisnis, Perikanan & Hayati','BUDIDAYA IKAN AIR TAWAR DI KOLAM':'Agrobisnis, Perikanan & Hayati',
 'BUDIDAYA IKAN AIR TAWAR DI MEDIA LAINNYA':'Agrobisnis, Perikanan & Hayati','BUDIDAYA IKAN AIR TAWAR DI KARAMBA':'Agrobisnis, Perikanan & Hayati',
 'BUDIDAYA IKAN AIR TAWAR DI KERAMBA JARING APUNG':'Agrobisnis, Perikanan & Hayati','BUDIDAYA IKAN BERSIRIP LAUT':'Agrobisnis, Perikanan & Hayati',
 'BUDIDAYA IKAN HIAS AIR LAUT':'Agrobisnis, Perikanan & Hayati','BUDIDAYA IKAN HIAS AIR TAWAR':'Agrobisnis, Perikanan & Hayati',
 'BUDIDAYA PERIKANAN':'Agrobisnis, Perikanan & Hayati','BUDIDAYA PERTANIAN':'Agrobisnis, Perikanan & Hayati',
 'JASA SARANA PRODUKSI BUDIDAYA IKAN AIR TAWAR':'Agrobisnis, Perikanan & Hayati','JASA SARANA PRODUKSI BUDIDAYA IKAN LAUT':'Agrobisnis, Perikanan & Hayati',
 'NELAYAN LAUT':'Agrobisnis, Perikanan & Hayati','OLAHAN PERIKANAN':'Agrobisnis, Perikanan & Hayati',
 'OLAHAN PERTANIAN':'Agrobisnis, Perikanan & Hayati','OLAHAN PETERNAKAN':'Agrobisnis, Perikanan & Hayati',
 'PEMBEKUAN IKAN':'Agrobisnis, Perikanan & Hayati','PENGASAPAN/PEMANGGANGAN IKAN':'Agrobisnis, Perikanan & Hayati',
 'PENGASINAN BUAH-BUAHAN DAN SAYURAN':'Agrobisnis, Perikanan & Hayati','PENGGARAMAN/PENGERINGAN IKAN':'Agrobisnis, Perikanan & Hayati',
 'PENGOLAHAN DAN PENGAWETAN PRODUK DAGING DAN DAGING UNGGAS':'Agrobisnis, Perikanan & Hayati',
 'PEMBENIHAN IKAN AIR TAWAR':'Agrobisnis, Perikanan & Hayati','PEMBENIHAN IKAN LAUT':'Agrobisnis, Perikanan & Hayati',
 'PEMBIBITAN DAN BUDIDAYA KAMBING PERAH':'Agrobisnis, Perikanan & Hayati','PEMBIBITAN TANAMAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN ALAT-ALAT PERTANIAN':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN BUAH-BUAHAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN BUNGA POTONG/FLORIST':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN DAGING DAN IKAN OLAHAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN HASIL PERIKANAN':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN HASIL PERTANIAN LAINNYA':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN HASIL PETERNAKAN':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN HEWAN PIARAAN (PET ANIMALS)':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN HEWAN TERNAK':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN IKAN HIAS':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR DAGING OLAHAN DAN IKAN OLAHAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI BUAHBUAHAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI HASIL PERIKANAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI HASIL PETERNAKAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI PADI DAN PALAWIJA':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI SAYUR-SAYURAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR KOMODITI TANAMAN HIAS DAN HASIL PERTANIAN LAINNYA':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR PAKAN TERNAK, PAKAN UNGGAS DAN PAKAN IKAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN KAKI LIMA DAN LOS PASAR PUPUK DAN PEMBERANTAS HAMA':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN PAKAN TERNAK/UNGGAS/IKAN DAN HEWAN PIARAAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN PERLENGKAPAN DAN MEDIA TANAMAN HIAS':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN PUPUK DAN PEMBERANTAS HAMA':'Agrobisnis, Perikanan & Hayati','PERDAGANGAN ECERAN SAYURAN':'Agrobisnis, Perikanan & Hayati',
 'PERDAGANGAN ECERAN TANAMAN DAN BIBIT TANAMAN':'Agrobisnis, Perikanan & Hayati','PERTANIAN PADI HIBRIDA':'Agrobisnis, Perikanan & Hayati',
 'PETERNAKAN':'Agrobisnis, Perikanan & Hayati','PRODUKSI KOMPOS SAMPAH ORGANIK':'Agrobisnis, Perikanan & Hayati',
 'RUMAH POTONG DAN PENGEPAKAN DAGING':'Agrobisnis, Perikanan & Hayati','RUMAH POTONG DAN PENGEPAKAN UNGGAS':'Agrobisnis, Perikanan & Hayati',
 'TANAMAN HIAS':'Agrobisnis, Perikanan & Hayati','URBAN FARMING BUAH':'Agrobisnis, Perikanan & Hayati','URBAN FARMING SAYUR':'Agrobisnis, Perikanan & Hayati',

 # Jasa, Teknologi & Otomotif
 'AGEN PERJALANAN IBADAH UMROH DAN HAJI KHUSUS':'Jasa, Teknologi & Otomotif','AGEN PERJALANAN WISATA':'Jasa, Teknologi & Otomotif',
 'AGEN WISATA PERJALANAN (AWP)':'Jasa, Teknologi & Otomotif','BIRO PERJALANAN WISATA':'Jasa, Teknologi & Otomotif',
 'ELEKTRONIK':'Jasa, Teknologi & Otomotif','HANDPHONE , TABLET DAN AKSESORIESNYA':'Jasa, Teknologi & Otomotif',
 'JASA FOTOKOPI, PENYIAPAN DOKUMEN DAN AKTIVITAS KHUSUS PENUNJANG KANTOR LAINNYA':'Jasa, Teknologi & Otomotif',
 'JASA KONSTRUKSI':'Jasa, Teknologi & Otomotif','KOMPUTER DAN PERLENGKAPANNYA':'Jasa, Teknologi & Otomotif',
 'LAUNDRY':'Jasa, Teknologi & Otomotif','OTOMOTIF':'Jasa, Teknologi & Otomotif','PANGKAS RAMBUT':'Jasa, Teknologi & Otomotif',
 'PEDAGANG KAKI LIMA DAN LOS PASAR/TOKO-ALAT FOTOGRAFI, ALAT OPTIK DAN PERLENGKAPANNYA':'Jasa, Teknologi & Otomotif',
 'PENDIDIKAN':'Jasa, Teknologi & Otomotif','PENYEWAAN ALAT PESTA':'Jasa, Teknologi & Otomotif',
 'PERDAGANGAN BESAR DAN ECERAN; REPARASI DAN PERAWATAN MOBIL DAN SEPEDA MOTOR':'Jasa, Teknologi & Otomotif',
 'PERLENGKAPAN KENDARAAN BERMOTOR':'Jasa, Teknologi & Otomotif','REPARASI FURNITUR DAN PERLENGKAPAN RUMAH':'Jasa, Teknologi & Otomotif',
 'REPARASI HANDPHONE':'Jasa, Teknologi & Otomotif','REPARASI KENDARAAN BERMOTOR':'Jasa, Teknologi & Otomotif',
 'REPARASI KOMPUTER DAN PERALATAN SEJENISNYA':'Jasa, Teknologi & Otomotif','REPARASI PERALATAN RUMAH TANGGA':'Jasa, Teknologi & Otomotif',
 'SALON KECANTIKAN':'Jasa, Teknologi & Otomotif','SERVICE ELEKTRONIK':'Jasa, Teknologi & Otomotif',
 'SERVICE KENDARAAN':'Jasa, Teknologi & Otomotif','SOFTWARE DAN ANTIVIRUS':'Jasa, Teknologi & Otomotif',
 'STEAM MOTOR / MOBIL':'Jasa, Teknologi & Otomotif','TATA RIAS PENGANTIN DAN SALON':'Jasa, Teknologi & Otomotif',
 'TEKNOLOGI':'Jasa, Teknologi & Otomotif','VIDEO GAME DAN PERLENGKAPANNYA':'Jasa, Teknologi & Otomotif',

 # Kriya & Industri Kreatif
 'DESIGN GRAFIS':'Kriya & Industri Kreatif','DESIGN INTERIOR':'Kriya & Industri Kreatif','FOTOGRAFI':'Kriya & Industri Kreatif',
 'KERAJINAN / SOUVENIR':'Kriya & Industri Kreatif','LUKISAN':'Kriya & Industri Kreatif',
 'PERLENGKAPAN DAPUR TERBUAT DARI KAYU, BAMBU ATAU ROTAN':'Kriya & Industri Kreatif','SABLON DAN DESIGN GRAFIS':'Kriya & Industri Kreatif',
 'SANGGAR TARI, TEATER, KESENIAN DAERAH':'Kriya & Industri Kreatif','SENI MUSIK / RUPA / TEATER / TARI':'Kriya & Industri Kreatif',
 'SUVENIR / KERAJINAN ATAU SUVENIR DARI KAYU, BAMBU, ROTAN, PANDAN, RUMPUT DAN SEJENISNYA':'Kriya & Industri Kreatif',
 'SUVENIR / KERAJINAN DARI KERAMIK':'Kriya & Industri Kreatif','SUVENIR / KERAJINAN DARI LOGAM':'Kriya & Industri Kreatif',
 'SUVENIR DARI KULIT, TULANG, TANDUK, GADING, BULU DAN BINATANG YANG DIAWETKAN':'Kriya & Industri Kreatif',
 'VIDEO & MUSIK':'Kriya & Industri Kreatif',
}

In [6]:
# Tambahkan kolom sektor_makro lewat pemetaan dictionary eksak
df_clean['sektor_makro'] = df_clean['jenis_usaha'].map(SEKTOR_MAP)

missing = df_clean[df_clean['sektor_makro'].isna()]['jenis_usaha'].unique()
print("jenis_usaha yang belum termapping (harus kosong):", missing)

# Jika ada nilai yang unmapped (mis. update dataset di masa depan), baris tsb dibuang
df_clean = df_clean.dropna(subset=['sektor_makro']).copy()

jenis_usaha yang belum termapping (harus kosong): <ArrowStringArray>
[]
Length: 0, dtype: str


## 5. Fitur 2 — `jenis_instansi_pembina` (normalisasi `skpd`)
Kolom `skpd` memuat 25 nama instansi pembina yang bervariasi teksnya antar wilayah (mis. "SUKU DINAS PERINDUSTRIAN, PERDAGANGAN, KOPERASI..." berbeda redaksi di tiap kota/kabupaten administrasi). Fungsi `instansi_type()` menormalisasi nama-nama tersebut ke dalam 6 kategori jenis instansi pembina berdasarkan kata kunci fungsional, sehingga bisa dipakai sebagai fitur kategorikal K-Modes yang konsisten lintas wilayah.

In [7]:
def instansi_type(skpd: str) -> str:
    s = str(skpd).upper()
    if 'PERINDUSTRIAN' in s or ('PERDAGANGAN' in s and 'KOPERASI' in s):
        return 'Sudin Perindustrian, Perdagangan & UKM'
    if 'KETAHANAN PANGAN' in s:
        return 'Sudin Ketahanan Pangan, Kelautan & Pertanian'
    if 'TENAGA KERJA' in s:
        return 'Sudin/Dinas Tenaga Kerja & Transmigrasi'
    if 'PEMBERDAYAAN' in s or 'PPA' in s:
        return 'Sudin PPA & Pengendalian Penduduk'
    if 'SOSIAL' in s:
        return 'Dinas Sosial'
    if 'PARIWISATA' in s:
        return 'Dinas Pariwisata & Ekonomi Kreatif'
    return 'Lainnya'

df_clean['jenis_instansi_pembina'] = df_clean['skpd'].apply(instansi_type)
print("=== DISTRIBUSI JENIS INSTANSI PEMBINA ===")
print(df_clean['jenis_instansi_pembina'].value_counts())

=== DISTRIBUSI JENIS INSTANSI PEMBINA ===
jenis_instansi_pembina
Sudin Perindustrian, Perdagangan & UKM          193012
Sudin Ketahanan Pangan, Kelautan & Pertanian     26752
Sudin/Dinas Tenaga Kerja & Transmigrasi          22972
Sudin PPA & Pengendalian Penduduk                11978
Dinas Sosial                                      6050
Dinas Pariwisata & Ekonomi Kreatif                5914
Name: count, dtype: int64


## 6. Fitur 3 — `kelompok_tahun_berdiri` (binning `tahun`)
Kolom `tahun` (tahun berdiri usaha) di-*binning* menjadi 3 kelompok umur usaha. Fitur ini hanya bermakna karena sumber data sudah diganti ke `periode_data == 2024`, yang merupakan snapshot kumulatif 2020-2024 dengan variasi `tahun` yang nyata (berbeda dari `periode_data == 2025` yang nilainya konstan).

In [8]:
def tahun_bin(t: int) -> str:
    if t >= 2024:
        return 'Rintisan Baru (2024)'
    elif t >= 2022:
        return 'Berkembang (2022-2023)'
    else:
        return 'Mapan (2020-2021)'

df_clean['kelompok_tahun_berdiri'] = df_clean['tahun'].apply(tahun_bin)
print("=== DISTRIBUSI KELOMPOK TAHUN BERDIRI ===")
print(df_clean['kelompok_tahun_berdiri'].value_counts())

=== DISTRIBUSI KELOMPOK TAHUN BERDIRI ===
kelompok_tahun_berdiri
Mapan (2020-2021)         166999
Berkembang (2022-2023)     68355
Rintisan Baru (2024)       31324
Name: count, dtype: int64


## 7. Eksplorasi Awal Distribusi Wilayah Administrasi
Mengecek distribusi populasi UMKM pada 6 wilayah administrasi di DKI Jakarta.

In [9]:
print("=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===")
display(df_clean['wilayah'].value_counts().reset_index())

print("\n=== 10 JENIS USAHA TERBANYAK DI DKI JAKARTA ===")
display(df_clean['jenis_usaha'].value_counts().head(10).reset_index())

=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===


,wilayah,count
0,KOTA ADM. JAKARTA SELATAN,69674
1,KOTA ADM. JAKARTA TIMUR,57841
2,KOTA ADM. JAKARTA BARAT,53910
3,KOTA ADM. JAKARTA UTARA,44356
4,KOTA ADM. JAKARTA PUSAT,37054
5,KAB. ADM. KEP. SERIBU,3843



=== 10 JENIS USAHA TERBANYAK DI DKI JAKARTA ===


,jenis_usaha,count
0,MAKANAN DAN MINUMAN,135293
1,PRODUK MAKANAN LAINNYA,24678
2,WARUNG SEMBAKO,17605
3,FASHION,16850
4,KERAJINAN / SOUVENIR,6439
5,KUE BASAH,6269
6,KEDAI MAKANAN,4519
7,OLAHAN PERTANIAN,3862
8,RUMAH/WARUNG MAKAN,3835
9,"WARUNG KOPI / KEDAI KOPI , KEDAI JUS DAN MINUMAN LAINNYA",3396


In [10]:
print("=== DISTRIBUSI POPULASI 6 SEKTOR MAKRO ===")
dist_sektor = df_clean['sektor_makro'].value_counts().reset_index()
dist_sektor.columns = ['Sektor Makro', 'Jumlah UMKM']
dist_sektor['Persentase (%)'] = (dist_sektor['Jumlah UMKM'] / len(df_clean)) * 100
display(dist_sektor)

print("\n=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===")
dist_wilayah = df_clean['wilayah'].value_counts().reset_index()
dist_wilayah.columns = ['Wilayah Administrasi', 'Jumlah UMKM']
dist_wilayah['Persentase (%)'] = (dist_wilayah['Jumlah UMKM'] / len(df_clean)) * 100
display(dist_wilayah)

=== DISTRIBUSI POPULASI 6 SEKTOR MAKRO ===


,Sektor Makro,Jumlah UMKM,Persentase (%)
0,Kuliner & Pangan Olahan,186244,69.838532
1,Fesyen & Industri Tekstil,23699,8.886747
2,Perdagangan Sembako & Retail,21730,8.148404
3,"Agrobisnis, Perikanan & Hayati",12597,4.723674
4,Kriya & Industri Kreatif,11586,4.344565
5,"Jasa, Teknologi & Otomotif",10822,4.058078



=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===


,Wilayah Administrasi,Jumlah UMKM,Persentase (%)
0,KOTA ADM. JAKARTA SELATAN,69674,26.126640
1,KOTA ADM. JAKARTA TIMUR,57841,21.689453
2,KOTA ADM. JAKARTA BARAT,53910,20.215391
3,KOTA ADM. JAKARTA UTARA,44356,16.632793
4,KOTA ADM. JAKARTA PUSAT,37054,13.894659
5,KAB. ADM. KEP. SERIBU,3843,1.441064


## 8. Penyimpanan Dataset Bersih Tahap 1
Menyimpan dataset yang telah dibersihkan, lengkap dengan 3 fitur kategorikal baru (`sektor_makro`, `jenis_instansi_pembina`, `kelompok_tahun_berdiri`), ke direktori `data/processed/` sebagai acuan untuk tahap pemodelan K-Modes.

In [11]:
FITUR_BARU = ['sektor_makro', 'jenis_instansi_pembina', 'kelompok_tahun_berdiri']
print("=== VALIDASI NILAI KOSONG PADA 3 FITUR BARU (harus 0) ===")
print(df_clean[FITUR_BARU].isnull().sum())

output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

clean_output_path = os.path.join(output_dir, '01_umkm_clean.csv')
df_clean.to_csv(clean_output_path, index=False)

print(f"\nDataset bersih berhasil disimpan di: {clean_output_path}")
print(f"Total baris: {len(df_clean):,} | Total kolom: {df_clean.shape[1]}")

=== VALIDASI NILAI KOSONG PADA 3 FITUR BARU (harus 0) ===
sektor_makro              0
jenis_instansi_pembina    0
kelompok_tahun_berdiri    0
dtype: int64

Dataset bersih berhasil disimpan di: ..\data\processed\01_umkm_clean.csv
Total baris: 266,678 | Total kolom: 10
